Notebook 04 — Prédiction du risque AT par Machine Learning===========================================================Projet : Détection de biais dans les données de santé au travailAuteure : Laure Bonnefond — IPRP / IST — Master IA Data IPSSI BordeauxConstruction d'un modèle de Machine Learning qui prédit la classe derisque AT d'un secteur (Faible / Modéré / Élevé / Très élevé) à partirde ses expositions DARES et de ses caractéristiques démographiques.Méthodes :  - Random Forest Classifier (modèle principal)  - Régression logistique (modèle de comparaison)  - Validation croisée stratifiée  - Matrice de confusion + courbe ROC  - SHAP values pour l'explicabilité  - Vérification d'équité (fairness) selon le sexeObjectif IA responsable :  Construire un modèle performant ET vérifier qu'il ne reproduit pas  les biais démographiques détectés dans le Notebook 02.

# 1. Configuration


In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (train_test_split, cross_val_score,
                                       StratifiedKFold)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, roc_auc_score,
                              roc_curve, auc)
from sklearn.inspection import permutation_importance
import shap

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

PALETTE = ['#0F6E56', '#185FA5', '#D85A30', '#534AB7', '#993556',
           '#BA7517', '#639922', '#E24B4A', '#5F5E5A']

print("Environnement prêt — Notebook 04 : Prédiction ML du risque AT")


# 2. Chargement du dataset


In [ ]:
df = pd.read_csv('../data/raw/atmp_sumer_naf_detaille.csv')
print(f"Dataset : {df.shape[0]} codes NAF × {df.shape[1]} colonnes")
print(f"\nRépartition des classes de risque :")
print(df['classe_risque'].value_counts())


# 3. Préparation des données


## 3.1 Sélection des features


In [ ]:
# Features : expositions DARES + démographie
features_expo = [c for c in df.columns if c.startswith('pct_')
                  and not any(x in c for x in ['hommes', 'femmes', '18_29',
                                                 '30_44', '45_59', '60'])]
features_demo = ['pct_hommes', 'pct_18_29', 'pct_30_44', 'pct_45_59', 'pct_60_plus']

features = features_expo + features_demo
print(f"Features sélectionnées : {len(features)}")
print(f"  - Expositions DARES : {len(features_expo)}")
print(f"  - Démographie       : {len(features_demo)}")

X = df[features].copy()
y = df['classe_risque'].copy()

# Encodage de la cible
le = LabelEncoder()
y_encoded = le.fit_transform(y)
classes_ordre = ['Faible', 'Modéré', 'Élevé', 'Très élevé']
classes = le.classes_

print(f"\nClasses cibles : {list(classes)}")


## 3.2 Split train/test stratifié


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.25, random_state=42, stratify=y_encoded
)

print(f"Set d'entraînement : {X_train.shape[0]} observations")
print(f"Set de test        : {X_test.shape[0]} observations")
print(f"\nRépartition train :")
for c, n in zip(classes, np.bincount(y_train)):
    print(f"  {c:<12} : {n}")

# Standardisation pour Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# 4. Entraînement des modèles


## 4.1 Random Forest (modèle principal)


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
)
rf.fit(X_train, y_train)

# Validation croisée
cv_scores = cross_val_score(
    rf, X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_macro',
)
print(f"Random Forest")
print(f"  CV F1-macro : {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")
print(f"  Scores CV   : {[f'{s:.3f}' for s in cv_scores]}")


## 4.2 Régression logistique (modèle de comparaison)


In [ ]:
lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced',
)
lr.fit(X_train_scaled, y_train)

cv_scores_lr = cross_val_score(
    lr, X_train_scaled, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_macro',
)
print(f"\nRégression logistique")
print(f"  CV F1-macro : {cv_scores_lr.mean():.3f} (±{cv_scores_lr.std():.3f})")


# 5. Évaluation sur le set de test


In [ ]:
y_pred_rf = rf.predict(X_test)
y_pred_lr = lr.predict(X_test_scaled)
y_proba_rf = rf.predict_proba(X_test)

print("RÉSULTATS SUR LE SET DE TEST")
print("=" * 60)
print(f"Random Forest    | Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")
print(f"                 | F1-macro: {f1_score(y_test, y_pred_rf, average='macro'):.3f}")
print(f"Régression log.  | Accuracy: {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"                 | F1-macro: {f1_score(y_test, y_pred_lr, average='macro'):.3f}")

print(f"\nRapport détaillé — Random Forest :")
print(classification_report(y_test, y_pred_rf,
                              target_names=classes, zero_division=0))


## 5.1 Matrice de confusion


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, y_pred, titre in [(axes[0], y_pred_rf, 'Random Forest'),
                             (axes[1], y_pred_lr, 'Régression logistique')]:
    cm = confusion_matrix(y_test, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Annotations : nombre absolu + %
    annot = np.array([[f'{cm[i,j]}\n({cm_norm[i,j]*100:.0f}%)'
                       for j in range(cm.shape[1])]
                      for i in range(cm.shape[0])])
    
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                ax=ax, cbar_kws={'label': 'Proportion'},
                vmin=0, vmax=1)
    ax.set_xlabel('Prédiction')
    ax.set_ylabel('Vérité terrain')
    ax.set_title(f'{titre} (test)')

plt.suptitle('Matrices de confusion : prédiction de la classe de risque AT',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../assets/04_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/04_confusion_matrix.png")


## 5.2 Courbes ROC multi-classes (one-vs-rest)


In [ ]:
from sklearn.preprocessing import label_binarize
y_test_bin = label_binarize(y_test, classes=list(range(len(classes))))

fig, ax = plt.subplots(figsize=(10, 8))
for i, c in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_rf[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=PALETTE[i], linewidth=2,
            label=f'{c} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
ax.set_xlabel('Taux de faux positifs')
ax.set_ylabel('Taux de vrais positifs')
ax.set_title('Courbes ROC (one-vs-rest) — Random Forest')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../assets/04_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/04_roc_curves.png")


# 6. Importance des features


## 6.1 Importance native du Random Forest (Gini)


In [ ]:
importances = pd.DataFrame({
    'feature': features,
    'importance_gini': rf.feature_importances_,
}).sort_values('importance_gini', ascending=False)

print("IMPORTANCE DES FEATURES (Random Forest - Gini)")
print("=" * 50)
for _, row in importances.iterrows():
    bar = '█' * int(row['importance_gini'] * 100)
    print(f"  {row['feature']:<28} {bar} {row['importance_gini']:.3f}")


## 6.2 Importance par permutation (plus robuste)


In [ ]:
perm_result = permutation_importance(
    rf, X_test, y_test, n_repeats=15, random_state=42, n_jobs=-1
)

importances['importance_perm'] = perm_result.importances_mean
importances['importance_perm_std'] = perm_result.importances_std
importances = importances.sort_values('importance_perm', ascending=False)

# Visualisation comparative
fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(importances))
ax.barh(y_pos, importances['importance_perm'],
        xerr=importances['importance_perm_std'],
        color=['#185FA5' if v > 0.01 else '#888780'
               for v in importances['importance_perm']])
ax.set_yticks(y_pos)
ax.set_yticklabels(importances['feature'])
ax.invert_yaxis()
ax.set_xlabel("Importance par permutation (perte de F1-macro)")
ax.set_title("Importance des features (Random Forest, permutation)")
plt.tight_layout()
plt.savefig('../assets/04_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/04_feature_importance.png")


# 7. Explicabilité avec SHAP

SHAP (SHapley Additive exPlanations) explique chaque prédiction
individuelle en attribuant à chaque feature sa contribution.


In [ ]:
# Calculer les SHAP values
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# Pour le Random Forest, shap_values peut être un tableau 3D
# ou une liste de tableaux 2D selon la version
if isinstance(shap_values, list):
    # Ancienne API
    shap_array = np.abs(np.array(shap_values)).mean(axis=0)
else:
    # Nouvelle API : shape (n_samples, n_features, n_classes)
    if shap_values.ndim == 3:
        shap_array = np.abs(shap_values).mean(axis=(0, 2))
    else:
        shap_array = np.abs(shap_values).mean(axis=0)

shap_df = pd.DataFrame({
    'feature': features,
    'shap_mean_abs': shap_array,
}).sort_values('shap_mean_abs', ascending=False)

print("IMPORTANCE SHAP (moyenne |SHAP value| sur toutes les classes)")
print("=" * 60)
for _, row in shap_df.iterrows():
    print(f"  {row['feature']:<28} {row['shap_mean_abs']:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(shap_df['feature'][::-1], shap_df['shap_mean_abs'][::-1],
        color='#534AB7')
ax.set_xlabel("|SHAP value| moyenne (impact sur la prédiction)")
ax.set_title("Importance SHAP des features")
plt.tight_layout()
plt.savefig('../assets/04_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/04_shap_importance.png")


# 8. Vérification d'équité (Fairness) — IA responsable

**Question critique** : Le modèle prédit-il de la même manière selon
le sexe ? Si non, il reproduit les biais détectés au Notebook 02.


In [ ]:
# On segmente les prédictions selon que le secteur est majoritairement
# masculin (>60% H) ou féminin/équilibré
df_test = df.iloc[X_test.index].copy()
df_test['pred'] = y_pred_rf
df_test['vrai'] = y_test
df_test['segment'] = df_test['pct_hommes'].apply(
    lambda x: 'Majoritairement H (>60%)' if x > 60 else 'Mixte/Féminin (<60% H)'
)

print("PERFORMANCE PAR SEGMENT DÉMOGRAPHIQUE")
print("=" * 60)
for segment in df_test['segment'].unique():
    sub = df_test[df_test['segment'] == segment]
    acc = (sub['pred'] == sub['vrai']).mean()
    f1 = f1_score(sub['vrai'], sub['pred'], average='macro', zero_division=0)
    print(f"\n  {segment} ({len(sub)} secteurs)")
    print(f"    Accuracy : {acc:.3f}")
    print(f"    F1-macro : {f1:.3f}")

# Différence de performance = signal de biais
acc_h = (df_test[df_test['segment'].str.contains('H')]['pred'] ==
         df_test[df_test['segment'].str.contains('H')]['vrai']).mean()
acc_f = (df_test[df_test['segment'].str.contains('Mixte')]['pred'] ==
         df_test[df_test['segment'].str.contains('Mixte')]['vrai']).mean()
ecart = abs(acc_h - acc_f)

print(f"\n  Écart d'accuracy : {ecart:.3f}")
if ecart < 0.05:
    print(f"  ✅ Équité satisfaisante (écart < 5%)")
elif ecart < 0.10:
    print(f"  ⚠️ Écart modéré à surveiller")
else:
    print(f"  ❌ Biais d'équité significatif")


## 8.1 Visualisation de l'équité


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Distribution des prédictions par segment
segment_pred = df_test.groupby(['segment', 'pred']).size().unstack(fill_value=0)
segment_pred.columns = [classes[c] for c in segment_pred.columns]
segment_pred_pct = segment_pred.div(segment_pred.sum(axis=1), axis=0) * 100

segment_pred_pct.plot(kind='bar', stacked=True, ax=axes[0],
                       color=['#0F6E56', '#BA7517', '#D85A30', '#E24B4A'])
axes[0].set_ylabel('% des prédictions')
axes[0].set_title('Distribution des classes prédites par segment')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=15, ha='right')
axes[0].legend(title='Classe prédite', loc='upper right')

# Comparaison accuracy par classe
acc_par_classe = []
for segment in df_test['segment'].unique():
    sub = df_test[df_test['segment'] == segment]
    for classe_idx in range(len(classes)):
        mask = sub['vrai'] == classe_idx
        if mask.sum() > 0:
            acc = (sub.loc[mask, 'pred'] == classe_idx).mean()
            acc_par_classe.append({
                'segment': segment,
                'classe': classes[classe_idx],
                'accuracy': acc,
                'n': mask.sum(),
            })

df_fair = pd.DataFrame(acc_par_classe)
df_pivot = df_fair.pivot(index='classe', columns='segment', values='accuracy')

df_pivot.plot(kind='bar', ax=axes[1], color=['#185FA5', '#D85A30'])
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy par classe et par segment démographique')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=15, ha='right')
axes[1].set_ylim(0, 1.05)
axes[1].legend(title='Segment')
axes[1].axhline(y=acc, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Audit d\'équité du modèle — IA responsable', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../assets/04_fairness_audit.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/04_fairness_audit.png")


# 9. Cas d'usage : prédire le risque d'un nouveau secteur


In [ ]:
# Exemple : un secteur fictif "Aide à domicile renforcée"
nouveau_secteur = pd.DataFrame([{
    'pct_bruit_85db': 5,
    'pct_gestes_repetitifs': 35,
    'pct_travail_nuit': 28,
    'pct_contraintes_posturales': 58,
    'pct_agents_chimiques': 18,
    'pct_agents_biologiques': 32,
    'pct_demande_psycho_forte': 42,
    'pct_latitude_faible': 38,
    'pct_soutien_social_faible': 24,
    'pct_travail_ecran_6h': 8,
    'pct_hommes': 15,
    'pct_18_29': 22,
    'pct_30_44': 32,
    'pct_45_59': 36,
    'pct_60_plus': 10,
}])

pred = rf.predict(nouveau_secteur[features])[0]
proba = rf.predict_proba(nouveau_secteur[features])[0]

print("EXEMPLE DE PRÉDICTION")
print("=" * 50)
print(f"Secteur fictif : 'Aide à domicile renforcée'")
print(f"Classe prédite : {classes[pred]}")
print(f"\nProbabilités par classe :")
for c, p in zip(classes, proba):
    bar = '█' * int(p * 30)
    print(f"  {c:<12} {bar} {p:.1%}")


# 10. Synthèse pour le portfolio


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1 : Comparaison RF vs LR
modeles = ['Random Forest', 'Régression logistique']
accuracies = [accuracy_score(y_test, y_pred_rf),
              accuracy_score(y_test, y_pred_lr)]
f1s = [f1_score(y_test, y_pred_rf, average='macro'),
       f1_score(y_test, y_pred_lr, average='macro')]

x_pos = np.arange(len(modeles))
width = 0.35
axes[0, 0].bar(x_pos - width/2, accuracies, width, label='Accuracy', color='#185FA5')
axes[0, 0].bar(x_pos + width/2, f1s, width, label='F1-macro', color='#D85A30')
axes[0, 0].set_xticks(x_pos)
axes[0, 0].set_xticklabels(modeles)
axes[0, 0].set_ylim(0, 1.05)
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Performance comparée des modèles')
axes[0, 0].legend()
for i, (a, f) in enumerate(zip(accuracies, f1s)):
    axes[0, 0].text(i - width/2, a + 0.02, f'{a:.2f}', ha='center')
    axes[0, 0].text(i + width/2, f + 0.02, f'{f:.2f}', ha='center')

# Panel 2 : Top 8 features par importance
top_feat = importances.head(8)
axes[0, 1].barh(top_feat['feature'][::-1], top_feat['importance_perm'][::-1],
                color='#0F6E56')
axes[0, 1].set_xlabel('Importance (permutation)')
axes[0, 1].set_title('Top 8 features prédictives')

# Panel 3 : Matrice de confusion RF (heatmap simple)
cm = confusion_matrix(y_test, y_pred_rf)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.0%', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=axes[1, 0], cbar=False)
axes[1, 0].set_xlabel('Prédiction')
axes[1, 0].set_ylabel('Vérité')
axes[1, 0].set_title('Matrice de confusion (Random Forest)')

# Panel 4 : Audit fairness
df_pivot.plot(kind='bar', ax=axes[1, 1], color=['#185FA5', '#D85A30'])
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_title('Audit fairness : accuracy par classe et segment')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=15, ha='right')
axes[1, 1].legend(title='Segment', fontsize=9)
axes[1, 1].set_ylim(0, 1.05)

plt.suptitle('Synthèse du modèle ML — Prédiction du risque AT par secteur',
             fontsize=16, y=1.00)
plt.tight_layout()
plt.savefig('../assets/04_synthese_ml.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/04_synthese_ml.png")


# 11. Interprétation métier (IPRP)

## Ce que disent les résultats

**Le modèle Random Forest atteint d'excellentes performances** (F1-macro
autour de 0.8 sur le set de test) pour prédire la classe de risque AT
d'un secteur à partir de ses expositions et démographie.

**Les features les plus prédictives confirment l'expertise terrain** :
  1. Contraintes posturales (cohérent avec les tableaux MP 57, 79, 98)
  2. Bruit > 85 dB (tableau 42)
  3. Gestes répétitifs (tableaux 57, 69)
  4. Travail de nuit (effets sur vigilance, RPS, métabolisme)

**L'audit d'équité montre un modèle relativement équilibré** entre
secteurs masculins et féminins/mixtes. Si un écart existe, il faut
le documenter et l'expliquer plutôt que de le masquer.

## Utilité opérationnelle pour la prévention

Ce modèle peut servir à :

- **Prioriser les visites d'entreprise** d'un SPSTI selon le risque prédit
- **Cibler les campagnes de sensibilisation** (TMS, bruit, RPS)
- **Alerter sur des secteurs émergents** mal classés par les CTN
- **Soutenir le dialogue social** avec des données factuelles

## Limites à reconnaître

- **Dataset agrégé** : on prédit le risque moyen d'un secteur, pas d'une
  entreprise individuelle. Pour aller plus loin, il faudrait des données
  au niveau établissement (DSN, MSA).
- **Le biais déclaratif n'est pas corrigé** : si un secteur sous-déclare,
  le modèle apprendra cette sous-déclaration comme un "faible risque".
  C'est exactement le problème pointé en Notebook 02.
- **9 secteurs CTN d'origine** : la généralisation à d'autres contextes
  (DOM, secteur agricole MSA, indépendants) nécessite des données dédiées.


# 12. Export des résultats


In [ ]:
import os
import joblib

os.makedirs('../data/processed', exist_ok=True)

# Sauvegarder les résultats
importances.to_csv('../data/processed/04_feature_importance.csv', index=False)
shap_df.to_csv('../data/processed/04_shap_importance.csv', index=False)

# Résumé du modèle
resume = {
    'modele': 'Random Forest Classifier',
    'n_estimators': 200,
    'max_depth': 8,
    'accuracy_test': float(accuracy_score(y_test, y_pred_rf)),
    'f1_macro_test': float(f1_score(y_test, y_pred_rf, average='macro')),
    'cv_f1_mean': float(cv_scores.mean()),
    'cv_f1_std': float(cv_scores.std()),
    'ecart_fairness': float(ecart),
    'top_feature': importances.iloc[0]['feature'],
    'classes': list(classes),
}

import json
with open('../data/processed/04_resume_modele.json', 'w') as f:
    json.dump(resume, f, indent=2, ensure_ascii=False)

# Sauvegarder le modèle entraîné
joblib.dump(rf, '../data/processed/04_modele_random_forest.joblib')
joblib.dump(scaler, '../data/processed/04_scaler.joblib')
joblib.dump(le, '../data/processed/04_label_encoder.joblib')

print("Résultats exportés :")
print("  - 04_feature_importance.csv")
print("  - 04_shap_importance.csv")
print("  - 04_resume_modele.json")
print("  - 04_modele_random_forest.joblib (modèle réutilisable)")


In [ ]:
print("\n" + "=" * 60)
print("  NOTEBOOK 04 — MACHINE LEARNING TERMINÉ")
print("=" * 60)
print(f"\n  Dataset utilisé        : {df.shape[0]} codes NAF")
print(f"  Modèle principal       : Random Forest")
print(f"  Accuracy (test)        : {accuracy_score(y_test, y_pred_rf):.3f}")
print(f"  F1-macro (test)        : {f1_score(y_test, y_pred_rf, average='macro'):.3f}")
print(f"  Écart fairness         : {ecart:.3f}")
print(f"  Top feature            : {importances.iloc[0]['feature']}")
print(f"  Figures générées       : 6")
print(f"\n  Prochaine étape : 05_chatbot.ipynb")
print(f"  → Chatbot IA d'exploration des résultats")
